# COOP Project 4 — All Models Kaggle Runtime

Notebook เดียวสำหรับ Demo โดยโหลด Model Runtime ทั้งหมดไว้ใน Kaggle session เดียว และบังคับใช้ CPU เท่านั้น

**Kaggle = Model Inference Only**

Backend ยังคงรับผิดชอบ Process Logic ทั้งหมด เช่น Template/ROI, Auto ROI, crop orchestration,
reading order, OCR normalize/merge, Table quality gate, semi-table/recovery, summary/KV,
SigLIP scoring/ranking/threshold และ Database/Business Logic

Endpoints:

- `/health`
- `/layout/health`, `/layout/predict`
- `/text-detection/health`, `/text-detection/predict`
- `/text-recognition/health`, `/text-recognition/predict`
- `/table/health`, `/table/predict`
- `/siglip/health`, `/siglip/predict`

Backend สามารถตั้ง URL ทั้ง 5 ตัวให้ชี้ Cloudflare base URL เดียวกัน แต่คนละ path

## CPU-only setting

Notebook นี้ตั้งใจให้ใช้ **CPU เท่านั้น**

ใน Kaggle ให้ตั้ง:

`Settings → Accelerator → None`

และ notebook จะตั้ง `CUDA_VISIBLE_DEVICES=""` เพื่อไม่ให้ Torch/Paddle ใช้ GPU แม้ environment จะมี GPU อยู่

ข้อควรทราบ: ทั้ง 5 โมเดลโหลดพร้อมกันได้ แต่ **Table / Layout / SigLIP อาจใช้เวลาประมวลผลค่อนข้างนานบน CPU** โดยเฉพาะรอบแรกที่โหลดโมเดล


In [1]:
# 1) Install dependencies
import sys
import subprocess

def pip_install(*packages):
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-cache-dir",
            *packages,
        ]
    )

# Core numeric dependency
pip_install(
    "numpy==2.3.3",
    "pillow==11.3.0",
)

# Paddle CPU engine
pip_install(
    "paddlepaddle==3.3.0",
)

# ให้ pip resolve PaddleOCR + PaddleX ที่เข้ากันเอง
pip_install(
    "paddleocr",
    "paddlex[ocr]==3.7.2",
)

# API
pip_install(
    "fastapi",
    "uvicorn[standard]",
    "python-multipart",
    "requests==2.32.4",
)

# SigLIP — install matching CPU PyTorch stack
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--no-cache-dir",
    "--index-url",
    "https://download.pytorch.org/whl/cpu",
    "torch==2.13.0",
    "torchvision==0.28.0",
])

pip_install(
    "transformers==5.16.1",
    "accelerate",
)

print("Dependencies installed.")
print("IMPORTANT: Restart Kaggle Session before continuing.")

Dependencies installed.
IMPORTANT: Restart Kaggle Session before continuing.


In [2]:
import numpy
import paddle
import paddleocr
import paddlex
import PIL
import torch
import transformers

print("NumPy:", numpy.__version__)
print("Paddle:", paddle.__version__)
print("Paddle device:", paddle.device.get_device())
print("PaddleOCR:", paddleocr.__version__)
print("PaddleX:", paddlex.__version__)
print("Pillow:", PIL.__version__)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)

import torchvision

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)

from transformers import SiglipModel, SiglipProcessor
print("SigLIP import: OK")

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


NumPy: 2.3.3
Paddle: 3.3.0
Paddle device: cpu
PaddleOCR: 3.7.0
PaddleX: 3.7.2
Pillow: 11.3.0
Torch: 2.13.0+cpu
Transformers: 5.16.1
Torch: 2.13.0+cpu
Torchvision: 0.28.0+cpu
CUDA available: False
Transformers: 5.16.1
SigLIP import: OK


In [3]:
# 2) Imports + JSON/Image utilities
# CPU ONLY — disable CUDA/GPU visibility before importing model frameworks.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["FLAGS_selected_gpus"] = ""
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = os.environ.get(
    "PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "False"
)
print("CPU-only mode enabled: CUDA_VISIBLE_DEVICES is empty")
import base64, io, json, math, os, re, tempfile, time, traceback
from typing import Any, Dict, List, Optional

import numpy as np
from PIL import Image

def jsonable(v):
    if v is None or isinstance(v, (str, int, bool)):
        return v
    if isinstance(v, float):
        return None if math.isnan(v) or math.isinf(v) else v
    if isinstance(v, np.generic):
        return jsonable(v.item())
    if isinstance(v, np.ndarray):
        return jsonable(v.tolist())
    if isinstance(v, dict):
        return {str(k): jsonable(x) for k, x in v.items()}
    if isinstance(v, (list, tuple, set)):
        return [jsonable(x) for x in v]

    for attr in ("numpy", "tolist"):
        try:
            fn = getattr(v, attr, None)
            if callable(fn):
                return jsonable(fn())
        except Exception:
            pass

    try:
        res = getattr(v, "res", None)
        if res is not None:
            return jsonable(res)
    except Exception:
        pass

    try:
        j = getattr(v, "json", None)
        if callable(j):
            return jsonable(j())
        if j is not None:
            return jsonable(j)
    except Exception:
        pass

    try:
        return jsonable(vars(v))
    except Exception:
        return str(v)

def decode_image(value: str) -> Image.Image:
    s = value.strip()
    if s.startswith("data:"):
        s = s.split(",", 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(s))).convert("RGB")

def mapping(obj):
    if isinstance(obj, dict):
        return obj
    try:
        if isinstance(obj.res, dict):
            return obj.res
    except Exception:
        pass
    x = jsonable(obj)
    if isinstance(x, dict):
        return x.get("res", x) if isinstance(x.get("res", x), dict) else x
    return {}

def items(output):
    if output is None: return []
    if isinstance(output, list): return output
    if isinstance(output, tuple): return list(output)
    if isinstance(output, dict): return [output]
    try: return list(output)
    except TypeError: return [output]

def paddle_predict(model, image):
    arr = np.asarray(image)
    try:
        return model.predict(arr)
    except Exception as e:
        print("ndarray predict failed; retry temp file:", repr(e))

    path = None
    try:
        with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as f:
            path = f.name
        image.save(path)
        return model.predict(path)
    finally:
        if path and os.path.exists(path):
            try: os.remove(path)
            except Exception: pass

print("Utilities ready.")

CPU-only mode enabled: CUDA_VISIBLE_DEVICES is empty
Utilities ready.


In [4]:
# 3) Load all models
# CPU ONLY: Kaggle Accelerator = None

import time
import traceback

MODELS = {}
LOAD_ERRORS = {}

def load_or_fail(name, loader):
    started = time.time()
    try:
        model = loader()
        MODELS[name] = model
        print(f"[OK] {name}: {time.time() - started:.2f}s")
        return model
    except Exception as e:
        MODELS[name] = None
        LOAD_ERRORS[name] = repr(e)
        print(f"[FAIL] {name}: {repr(e)}")
        traceback.print_exc()
        raise


# -------------------------------------------------
# Paddle models
# -------------------------------------------------

def load_layout():
    from paddleocr import LayoutDetection

    return LayoutDetection(
        model_name="PP-DocLayoutV3",
        device="cpu",
        enable_mkldnn=False,
    )


def load_text_detection():
    from paddleocr import TextDetection

    return TextDetection(
        model_name="PP-OCRv5_server_det",
        device="cpu",
        enable_mkldnn=False,
    )


def load_text_recognition():
    from paddleocr import TextRecognition

    return TextRecognition(
        model_name="th_PP-OCRv5_mobile_rec",
        device="cpu",
        enable_mkldnn=False,
    )


def load_table():
    from paddleocr import TableRecognitionPipelineV2

    return TableRecognitionPipelineV2(
        device="cpu",
        enable_mkldnn=False,
    )


# -------------------------------------------------
# SigLIP
# -------------------------------------------------

SIGLIP_PROCESSOR = None
SIGLIP_DEVICE = "cpu"


def load_siglip():
    global SIGLIP_PROCESSOR

    import torch
    from transformers import SiglipModel, SiglipProcessor

    model_name = "google/siglip-base-patch16-224"

    SIGLIP_PROCESSOR = SiglipProcessor.from_pretrained(
        model_name
    )

    model = SiglipModel.from_pretrained(
        model_name
    )

    model = model.to("cpu")
    model.eval()

    return model


# -------------------------------------------------
# LOAD
# -------------------------------------------------

# Paddle ต้องโหลดบน clean session
# ถ้าตัวแรกพัง ให้หยุดทันที แล้ว Restart Session
load_or_fail("layout", load_layout)

load_or_fail(
    "text_detection",
    load_text_detection
)

load_or_fail(
    "text_recognition",
    load_text_recognition
)

load_or_fail(
    "table",
    load_table
)

# SigLIP แยกจาก Paddle
load_or_fail(
    "siglip",
    load_siglip
)


print("\nLoaded:")
for name, model in MODELS.items():
    print(
        f"  {name}:",
        model is not None
    )

if LOAD_ERRORS:
    print("\nErrors:")
    for name, err in LOAD_ERRORS.items():
        print(f"  {name}: {err}")
else:
    print("\nAll models loaded successfully.")

Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-DocLayoutV3`.
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv5_server_det`.


[OK] layout: 1.24s


Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/th_PP-OCRv5_mobile_rec`.


[OK] text_detection: 0.54s


Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/UVDoc`.


...

Creating model: ('PP-DocLayout-L', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-DocLayout-L`.
Creating model: ('PP-LCNet_x1_0_table_cls', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_table_cls`.
Creating model: ('SLANeXt_wired', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/SLANeXt_wired`.
Creating model: ('SLANeXt_wireless', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/SLANeXt_wireless`.
Creating model: ('RT-DETR-L_wired_table_cell_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/R

[OK] table: 8.76s


[2026-09-02 01:59:38,322] [    INFO] _client.py:1025 - HTTP Request: GET https://huggingface.co/api/models/google/siglip-base-patch16-224/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
[2026-09-02 01:59:38,526] [    INFO] _client.py:1025 - HTTP Request: HEAD https://huggingface.co/google/siglip-base-patch16-224/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
[2026-09-02 01:59:38,747] [    INFO] _client.py:1025 - HTTP Request: HEAD https://huggingface.co/google/siglip-base-patch16-224/resolve/main/chat_template.json "HTTP/1.1 404 Not Found"
[2026-09-02 01:59:38,942] [    INFO] _client.py:1025 - HTTP Request: HEAD https://huggingface.co/google/siglip-base-patch16-224/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
[2026-09-02 01:59:39,169] [    INFO] _client.py:1025 - HTTP Request: HEAD https://huggingface.co/google/siglip-base-patch16-224/resolve/main/audio_tokenizer_config.json "HTTP/1.1 404 Not Found"
[2026-09-02 01:59:3

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

[OK] siglip: 4.05s

Loaded:
  layout: True
  text_detection: True
  text_recognition: True
  table: True
  siglip: True

All models loaded successfully.


In [5]:
# 4) Inference adapters — Model inference/serialization only

import numpy as np


# -------------------------------------------------
# Helpers
# -------------------------------------------------

def first_not_none(*values):
    """
    Return ค่าแรกที่ไม่ใช่ None
    ห้ามใช้ `or` กับ numpy.ndarray เพราะจะเกิด:
    The truth value of an array with more than one element is ambiguous
    """
    for value in values:
        if value is not None:
            return value
    return None


def safe_jsonable(value):
    """
    Convert model outputs ให้เป็น JSON-safe Python objects
    """
    if value is None:
        return None

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, dict):
        return {
            str(k): safe_jsonable(v)
            for k, v in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            safe_jsonable(v)
            for v in value
        ]

    # เผื่อ jsonable เดิมรองรับ Paddle/Torch object
    try:
        return jsonable(value)
    except Exception:
        return value



TABLE_GEOMETRY_KEYS = {
    "table_structured",
    "cells",
    "table_cells",
    "cell_bbox",
    "cell_bboxes",
    "boxes",
    "bbox",
    "box",
    "polygon",
    "polygons",
    "cell_box",
    "cell_boxes",
    "cell_box_list",
    "table_res_list",
    "structure",
    "html",
    "pred_html",
    "table_html",
    "structure_html",
}


def collect_table_geometry(value, path="raw_output"):
    """
    Preserve raw SLANeXt/Paddle table geometry for the backend.
    This is serialization only: no quality gate, recovery, OCR assignment,
    or table post-processing runs in Kaggle Runtime.
    """
    found = {
        "table_structured": None,
        "cell_bbox": None,
        "boxes": None,
        "polygons": None,
        "html": None,
        "geometry_sources": [],
    }

    def remember(name, item, item_path):
        if item is None:
            return
        if name == "table_structured" and isinstance(item, dict):
            if found["table_structured"] is None:
                found["table_structured"] = item
        elif name in {"cells", "table_cells", "cell_bbox", "cell_bboxes", "cell_box", "cell_boxes", "cell_box_list"}:
            if found["cell_bbox"] is None and isinstance(item, list):
                found["cell_bbox"] = item
        elif name in {"boxes", "bbox", "box"}:
            if found["boxes"] is None:
                found["boxes"] = item
        elif name in {"polygon", "polygons"}:
            if found["polygons"] is None:
                found["polygons"] = item
        elif name in {"html", "pred_html", "table_html", "structure_html"}:
            if found["html"] is None and isinstance(item, str):
                found["html"] = item
        found["geometry_sources"].append(item_path)

    def walk(node, node_path):
        if isinstance(node, dict):
            for key, item in node.items():
                key_name = str(key)
                item_path = f"{node_path}.{key_name}"
                if key_name in TABLE_GEOMETRY_KEYS:
                    remember(key_name, item, item_path)
                walk(item, item_path)
        elif isinstance(node, list):
            for index, item in enumerate(node):
                walk(item, f"{node_path}[{index}]")

    walk(value, path)
    return {key: val for key, val in found.items() if val is not None or key == "geometry_sources"}

# -------------------------------------------------
# Layout
# -------------------------------------------------

def infer_layout(image):
    out = paddle_predict(
        MODELS["layout"],
        image
    )

    detections = []

    for obj in items(out):
        m = mapping(obj)

        boxes = m.get("boxes")

        layout_det_res = m.get("layout_det_res")
        if boxes is None and isinstance(layout_det_res, dict):
            boxes = layout_det_res.get("boxes")

        boxes = safe_jsonable(boxes)

        # -----------------------------------------
        # Case 1: output contains list of boxes
        # -----------------------------------------
        if isinstance(boxes, list):
            for b in boxes:
                if not isinstance(b, dict):
                    continue

                bbox = first_not_none(
                    b.get("bbox"),
                    b.get("coordinate"),
                    b.get("box"),
                )

                label = first_not_none(
                    b.get("label"),
                    b.get("category"),
                    b.get("type"),
                )

                score = first_not_none(
                    b.get("score"),
                    b.get("confidence"),
                )

                detections.append({
                    "bbox": safe_jsonable(bbox),
                    "label": safe_jsonable(label),
                    "score": safe_jsonable(score),
                    "cls_id": safe_jsonable(
                        b.get("cls_id")
                    ),
                    "polygon_points": safe_jsonable(
                        b.get("polygon_points")
                    ),
                })

        # -----------------------------------------
        # Case 2: object itself is one detection
        # -----------------------------------------
        else:
            bbox = first_not_none(
                m.get("bbox"),
                m.get("coordinate"),
                m.get("box"),
            )

            label = first_not_none(
                m.get("label"),
                m.get("category"),
                m.get("type"),
            )

            score = first_not_none(
                m.get("score"),
                m.get("confidence"),
            )

            if bbox is not None or label is not None:
                detections.append({
                    "bbox": safe_jsonable(bbox),
                    "label": safe_jsonable(label),
                    "score": safe_jsonable(score),
                })

    return {
        "detections": detections
    }


# -------------------------------------------------
# Text Detection
# -------------------------------------------------

def infer_text_detection(image):
    out = paddle_predict(
        MODELS["text_detection"],
        image
    )

    polys = []
    scores = []

    for obj in items(out):
        m = mapping(obj)

        # สำคัญ:
        # ห้ามเขียน:
        # m.get("dt_polys") or m.get("polys")
        # เพราะ dt_polys อาจเป็น numpy.ndarray

        p = first_not_none(
            m.get("dt_polys"),
            m.get("polys"),
            m.get("polygons"),
            m.get("boxes"),
        )

        s = first_not_none(
            m.get("dt_scores"),
            m.get("scores"),
            m.get("confidence"),
        )

        # -----------------------------------------
        # polygons
        # -----------------------------------------
        if p is not None:
            p = safe_jsonable(p)

            if isinstance(p, list):

                # Single polygon
                #
                # [
                #   [x1, y1],
                #   [x2, y2],
                #   [x3, y3],
                #   [x4, y4]
                # ]
                is_single_polygon = (
                    len(p) > 0
                    and isinstance(p[0], list)
                    and len(p[0]) > 0
                    and isinstance(
                        p[0][0],
                        (int, float)
                    )
                )

                if is_single_polygon:
                    polys.append(p)

                # Multiple polygons
                #
                # [
                #   [[x,y], ...],
                #   [[x,y], ...]
                # ]
                else:
                    for poly in p:
                        if poly is not None:
                            polys.append(poly)

        # -----------------------------------------
        # scores
        # -----------------------------------------
        if s is not None:
            s = safe_jsonable(s)

            if isinstance(s, list):
                scores.extend(s)
            else:
                scores.append(s)

    return {
        "dt_polys": polys,
        "dt_scores": scores,
    }


# -------------------------------------------------
# Text Recognition
# -------------------------------------------------

def extract_rec(obj):
    m = mapping(obj)

    text = first_not_none(
        m.get("rec_text"),
        m.get("text"),
        m.get("label"),
    )

    score = first_not_none(
        m.get("rec_score"),
        m.get("score"),
        m.get("confidence"),
    )

    return {
        "rec_text": (
            ""
            if text is None
            else str(safe_jsonable(text))
        ),
        "rec_score": safe_jsonable(score),
    }


def infer_text_recognition(image):
    out = items(
        paddle_predict(
            MODELS["text_recognition"],
            image
        )
    )

    if not out:
        return {
            "rec_text": "",
            "rec_score": None,
        }

    return extract_rec(out[0])


def infer_text_recognition_batch(images):
    # sequential เพื่อรักษาลำดับ crop
    results = []

    for img in images:
        results.append(
            infer_text_recognition(img)
        )

    return {
        "results": results
    }


# -------------------------------------------------
# Table
# -------------------------------------------------

def infer_table(image):
    path = None
    try:
        with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as f:
            path = f.name
        image.save(path)
        out = MODELS["table"].predict(
            input=path,
            use_doc_orientation_classify=False,
            use_doc_unwarping=False,
            use_layout_detection=False,
            use_ocr_model=True,
        )
    finally:
        if path and os.path.exists(path):
            try:
                os.remove(path)
            except Exception:
                pass

    raw = items(out)
    raw_json = safe_jsonable(raw)
    geometry = collect_table_geometry(raw_json)

    response = {
        "raw_output": raw_json,
        "geometry_sources": geometry.get("geometry_sources", []),
    }

    for key in ("table_structured", "cell_bbox", "boxes", "polygons", "html"):
        if geometry.get(key) is not None:
            response[key] = geometry[key]

    print("===== TABLE RAW OUTPUT =====")
    import json
    print(
        json.dumps(
            response,
            ensure_ascii=False,
            indent=2
        )[:30000]
    )

    return response


# -------------------------------------------------
# SigLIP
# -------------------------------------------------

def infer_siglip(image, categories):
    if not categories:
        raise ValueError(
            "categories is required"
        )

    import torch

    inputs = SIGLIP_PROCESSOR(
        text=list(categories),
        images=image,
        padding="max_length",
        return_tensors="pt",
    )

    inputs = {
        k: (
            v.to(SIGLIP_DEVICE)
            if hasattr(v, "to")
            else v
        )
        for k, v in inputs.items()
    }

    with torch.no_grad():
        out = MODELS["siglip"](
            **inputs
        )

    logits = getattr(
        out,
        "logits_per_image",
        None
    )

    if logits is None:
        logits = getattr(
            out,
            "logits",
            None
        )

    if logits is None:
        raise RuntimeError(
            "SigLIP output has no logits"
        )

    logits = safe_jsonable(logits)

    # SigLIP ปกติคืน [[score1, score2, ...]]
    # เอา outer batch dimension ออก
    if (
        isinstance(logits, list)
        and len(logits) == 1
        and isinstance(logits[0], list)
    ):
        logits = logits[0]

    return {
        "logits": logits,
        "categories": list(categories),
    }


print("Inference adapters ready.")

Inference adapters ready.


In [6]:
# 5) FastAPI endpoints
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional, List
import time
import traceback

app = FastAPI(
    title="COOP Project 4 - Kaggle All Models Runtime",
    version="2.0"
)

MODEL_LABELS = {
    "layout": "PP-DocLayoutV3",
    "text_detection": "PP-OCRv5_server_det",
    "text_recognition": "th_PP-OCRv5_mobile_rec",
    "table": "TableRecognitionPipelineV2",
    "siglip": "SigLIP",
}


# =========================================================
# Request Models
# =========================================================

class ImageRequest(BaseModel):
    image: str


class RecognitionRequest(BaseModel):
    image: Optional[str] = None
    images: Optional[List[str]] = None


class SiglipCategory(BaseModel):
    value: str
    label: Optional[str] = None
    prompt: str

    match_threshold: Optional[float] = None
    margin_threshold: Optional[float] = None
    evidence_temperature: Optional[float] = None

    enabled: bool = True


class SiglipRequest(BaseModel):
    image: str
    categories: List[SiglipCategory]


# =========================================================
# Common Wrapper
# =========================================================

def wrap(kind, fn):
    if MODELS.get(kind) is None:
        raise HTTPException(
            503,
            detail={
                "success": False,
                "model": MODEL_LABELS[kind],
                "error": LOAD_ERRORS.get(
                    kind,
                    "model unavailable",
                ),
            },
        )

    started = time.time()

    try:
        result = fn()

        return {
            "success": True,
            "model": MODEL_LABELS[kind],
            "result": result,
            "meta": {
                "elapsed_ms": round(
                    (time.time() - started) * 1000,
                    2,
                )
            },
        }

    except Exception as e:
        traceback.print_exc()

        raise HTTPException(
            500,
            detail={
                "success": False,
                "model": MODEL_LABELS[kind],
                "error": str(e),
            },
        )


# =========================================================
# Health
# =========================================================

@app.get("/health")
def health_all():
    return {
        "success": True,
        "models": {
            k: {
                "ready": MODELS.get(k) is not None,
                "model": MODEL_LABELS[k],
                "error": LOAD_ERRORS.get(k),
            }
            for k in MODEL_LABELS
        },
    }


def model_health(kind):
    ready = MODELS.get(kind) is not None

    return {
        "success": ready,
        "model": MODEL_LABELS[kind],
        "ready": ready,
        "error": LOAD_ERRORS.get(kind),
    }


@app.get("/layout/health")
def layout_health():
    return model_health("layout")


@app.get("/text-detection/health")
def text_det_health():
    return model_health("text_detection")


@app.get("/text-recognition/health")
def text_rec_health():
    return model_health("text_recognition")


@app.get("/table/health")
def table_health():
    return model_health("table")


@app.get("/siglip/health")
def siglip_health():
    return model_health("siglip")


# =========================================================
# Predict
# =========================================================

@app.post("/layout/predict")
def layout_predict(p: ImageRequest):
    return wrap(
        "layout",
        lambda: infer_layout(
            decode_image(p.image)
        ),
    )


@app.post("/text-detection/predict")
def text_det_predict(p: ImageRequest):
    return wrap(
        "text_detection",
        lambda: infer_text_detection(
            decode_image(p.image)
        ),
    )


@app.post("/text-recognition/predict")
def text_rec_predict(p: RecognitionRequest):

    if p.images:
        return wrap(
            "text_recognition",
            lambda: infer_text_recognition_batch(
                [decode_image(x) for x in p.images]
            ),
        )

    if p.image:
        return wrap(
            "text_recognition",
            lambda: infer_text_recognition(
                decode_image(p.image)
            ),
        )

    raise HTTPException(
        422,
        detail="image or images is required",
    )


@app.post("/table/predict")
def table_predict(p: ImageRequest):
    return wrap(
        "table",
        lambda: infer_table(
            decode_image(p.image)
        ),
    )


@app.post("/siglip/predict")
def siglip_predict(p: SiglipRequest):

    # Backend เป็นเจ้าของ category config / threshold
    # Runtime ใช้เฉพาะ prompt สำหรับ SigLIP inference
    enabled_categories = [
        category
        for category in p.categories
        if category.enabled
    ]

    if not enabled_categories:
        raise HTTPException(
            422,
            detail="At least one enabled SigLIP category is required",
        )

    prompts = [
        category.prompt
        for category in enabled_categories
    ]

    return wrap(
        "siglip",
        lambda: infer_siglip(
            decode_image(p.image),
            prompts,
        ),
    )


print("FastAPI routes ready.")

FastAPI routes ready.


In [7]:
# 6) Start Uvicorn
import threading, requests, time

HOST = "0.0.0.0"
PORT = 8000

def run_server():
    import uvicorn
    uvicorn.run(app, host=HOST, port=PORT, log_level="info")

threading.Thread(target=run_server, daemon=True).start()

for _ in range(60):
    try:
        r = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=1)
        if r.ok:
            print(json.dumps(r.json(), indent=2, ensure_ascii=False))
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError("FastAPI server did not start")

INFO:     Started server process [4893]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:58278 - "GET /health HTTP/1.1" 200 OK
{
  "success": true,
  "models": {
    "layout": {
      "ready": true,
      "model": "PP-DocLayoutV3",
      "error": null
    },
    "text_detection": {
      "ready": true,
      "model": "PP-OCRv5_server_det",
      "error": null
    },
    "text_recognition": {
      "ready": true,
      "model": "th_PP-OCRv5_mobile_rec",
      "error": null
    },
    "table": {
      "ready": true,
      "model": "TableRecognitionPipelineV2",
      "error": null
    },
    "siglip": {
      "ready": true,
      "model": "SigLIP",
      "error": null
    }
  }
}


In [8]:
# 7) Cloudflare Quick Tunnel — Demo URL
import platform, subprocess, os, re, time

cloudflared = "/kaggle/working/cloudflared"

if not os.path.exists(cloudflared):
    machine = platform.machine().lower()
    if machine in {"x86_64", "amd64"}:
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    elif machine in {"aarch64", "arm64"}:
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-arm64"
    else:
        raise RuntimeError(f"Unsupported architecture: {machine}")

    subprocess.check_call(["wget", "-q", "-O", cloudflared, url])
    os.chmod(cloudflared, 0o755)

proc = subprocess.Popen(
    [cloudflared, "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

PUBLIC_URL = None
pattern = re.compile(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com")
deadline = time.time() + 90

while time.time() < deadline:
    line = proc.stdout.readline()
    if not line:
        time.sleep(.2)
        continue
    print(line.rstrip())
    m = pattern.search(line)
    if m:
        PUBLIC_URL = m.group(0)
        break

if not PUBLIC_URL:
    raise RuntimeError("Cloudflare URL not found")

print("\n" + "="*72)
print("ALL MODEL RUNTIME READY")
print("BASE URL:", PUBLIC_URL)
print()
print("LAYOUT_MODEL_URL=" + PUBLIC_URL + "/layout")
print("TEXT_DETECTION_MODEL_URL=" + PUBLIC_URL + "/text-detection")
print("TEXT_RECOGNITION_MODEL_URL=" + PUBLIC_URL + "/text-recognition")
print("TABLE_MODEL_URL=" + PUBLIC_URL + "/table")
print("IMAGE_VERIFICATION_MODEL_URL=" + PUBLIC_URL + "/siglip")
print("="*72)

2026-09-02T01:59:43Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-02T01:59:43Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-02T01:59:47Z INF +--------------------------------------------------------------------------------------------+
2026-09-02T01:59:47Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-02T01:59:47Z INF |  https://bargain-findlaw-pixels-shannon.trycloudflare.

In [9]:
# 8) Remote health smoke test
import requests, json

paths = [
    "/health",
    "/layout/health",
    "/text-detection/health",
    "/text-recognition/health",
    "/table/health",
    "/siglip/health",
]

for path in paths:
    try:
        r = requests.get(PUBLIC_URL + path, timeout=30)
        print(path, r.status_code, json.dumps(r.json(), ensure_ascii=False)[:500])
    except Exception as e:
        print(path, "ERROR", repr(e))

/health ERROR ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'bargain-findlaw-pixels-shannon.trycloudflare.com\', port=443): Max retries exceeded with url: /health (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7c706067ae40>: Failed to resolve \'bargain-findlaw-pixels-shannon.trycloudflare.com\' ([Errno -2] Name or service not known)"))'))
/layout/health ERROR ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'bargain-findlaw-pixels-shannon.trycloudflare.com\', port=443): Max retries exceeded with url: /layout/health (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7c706067b050>: Failed to resolve \'bargain-findlaw-pixels-shannon.trycloudflare.com\' ([Errno -2] Name or service not known)"))'))
/text-detection/health ERROR ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'bargain-findlaw-pixels-shannon.trycloudflare.com\', port=443): Max retries exceeded with url: /text-detection/health (Caused by N

## Backend `.env`

หลังรัน Cloudflare cell ให้นำค่าที่ notebook print ออกมาใส่ Local Backend:

```env
LAYOUT_MODEL_URL=https://...trycloudflare.com/layout
TEXT_DETECTION_MODEL_URL=https://...trycloudflare.com/text-detection
TEXT_RECOGNITION_MODEL_URL=https://...trycloudflare.com/text-recognition
TABLE_MODEL_URL=https://...trycloudflare.com/table
IMAGE_VERIFICATION_MODEL_URL=https://...trycloudflare.com/siglip
```

Backend client สามารถต่อ `/predict` และ `/health` จาก base URL ของแต่ละ model ได้ตาม contract ที่กำหนดไว้

### หมายเหตุ

Notebook นี้ตั้งใจสำหรับ **Demo แบบ CPU-only** และโหลดทุก model ใน Kaggle session เดียว จึงสะดวกกว่าเปิด 5 sessions
แต่ใช้ RAM มากกว่าและ inference จะช้ากว่า GPU หาก model ใดโหลดไม่สำเร็จ `/health` จะแสดงสถานะของ model นั้นโดยไม่ทำให้ API ของ model อื่นหายไป

หาก TableRecognitionPipelineV2 API ของ PaddleOCR/PaddleX version ที่ Kaggle ใช้ต่างออกไป
ให้ปรับเฉพาะ `load_table()` / serialization ของ raw model result โดยไม่ย้าย Table Process จาก Backend เข้ามา